In [ ]:
import gmsh
import numpy as np

gmsh.initialize()

gmsh.model.add("three_phase_stator")

# --------------------------------------------------
# Parameters
# --------------------------------------------------

L = 2.0

r_outer = 0.80
r_inner = 0.35

r_wire = 0.05
r_slot = 0.58

lc_air = 0.06
lc_iron = 0.03
lc_wire = 0.015

# --------------------------------------------------
# Geometry
# --------------------------------------------------

square = gmsh.model.occ.addRectangle(
    -L/2,
    -L/2,
    0,
    L,
    L
)

outer = gmsh.model.occ.addDisk(
    0,
    0,
    0,
    r_outer,
    r_outer
)

inner = gmsh.model.occ.addDisk(
    0,
    0,
    0,
    r_inner,
    r_inner
)

wire_tags = []

for k in range(6):

    theta = 2*np.pi*k/6

    x = r_slot*np.cos(theta)
    y = r_slot*np.sin(theta)

    wire = gmsh.model.occ.addDisk(
        x,
        y,
        0,
        r_wire,
        r_wire
    )

    wire_tags.append(wire)

# --------------------------------------------------
# Fragment everything
# --------------------------------------------------

objects = [
    (2, square),
    (2, outer),
    (2, inner),
]

objects += [(2, w) for w in wire_tags]

gmsh.model.occ.fragment(objects, [])

gmsh.model.occ.synchronize()

# --------------------------------------------------
# Outer boundary (Dirichlet boundary)
# --------------------------------------------------

tol = 1e-8

outer_boundary = []

for dim, tag in gmsh.model.getEntities(1):

    xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)

    if (
        abs(xmin + L/2) < tol or
        abs(xmax - L/2) < tol or
        abs(ymin + L/2) < tol or
        abs(ymax - L/2) < tol
    ):
        outer_boundary.append(tag)

gmsh.model.addPhysicalGroup(
    1,
    outer_boundary,
    name="boundary"
)

# --------------------------------------------------
# Expected areas
# --------------------------------------------------

A_wire = np.pi * r_wire**2
A_inner = np.pi * r_inner**2
A_iron = np.pi * (r_outer**2 - r_inner**2) - 6*A_wire
A_outer = L**2 - np.pi * r_outer**2

tol = 1e-6

wire_domains = []
iron = []
air_inner = []
air_outer = []

print("Detected surfaces")
print("-----------------")

for dim, tag in gmsh.model.getEntities(2):

    area = gmsh.model.occ.getMass(dim, tag)

    print(f"tag={tag:2d} area={area:.8f}")

    if abs(area - A_wire) < tol:

        wire_domains.append(tag)

    elif abs(area - A_inner) < tol:

        air_inner.append(tag)

    elif abs(area - A_iron) < tol:

        iron.append(tag)

    elif abs(area - A_outer) < tol:

        air_outer.append(tag)

print()
print("Outer air :", air_outer)
print("Inner air :", air_inner)
print("Iron      :", iron)
print("Wires     :", wire_domains)

# --------------------------------------------------
# Physical groups
# --------------------------------------------------

gmsh.model.addPhysicalGroup(
    2,
    air_outer,
    name="air_outer"
)

gmsh.model.addPhysicalGroup(
    2,
    air_inner,
    name="air_inner"
)

gmsh.model.addPhysicalGroup(
    2,
    iron,
    name="iron"
)

for i, tag in enumerate(wire_domains):

    gmsh.model.addPhysicalGroup(
        2,
        [tag],
        name=f"phase_{i+1}"
    )

# --------------------------------------------------
# Mesh sizes
# --------------------------------------------------

gmsh.model.mesh.setSize(
    gmsh.model.getEntities(0),
    lc_air
)

for tag in iron:

    boundary = gmsh.model.getBoundary(
        [(2, tag)],
        recursive=True
    )

    gmsh.model.mesh.setSize(
        boundary,
        lc_iron
    )

for tag in wire_domains:

    boundary = gmsh.model.getBoundary(
        [(2, tag)],
        recursive=True
    )

    gmsh.model.mesh.setSize(
        boundary,
        lc_wire
    )

# --------------------------------------------------
# Generate mesh
# --------------------------------------------------

gmsh.model.mesh.generate(2)

gmsh.write("three_phase_stator.msh")

gmsh.fltk.run()

gmsh.finalize()

Exception: Unknown model face with tag 1